# Module 4 - Q&A RAG Pipeline
### RAG-Based Mental Health Support Chatbot

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline using:
- **Dataset**: Mental Health Counseling Conversations
- **Embeddings**: Sentence Transformers (all-MiniLM-L6-v2)
- **Vector Database**: Qdrant Cloud (free tier)
- **LLM**: Groq API

Kernel: Python (nlp-rag-clean)
Do not run any pip install or uninstall cells in this notebook.

---
## 1 - Setup

### Step 1.1 - Verify Environment
To confirm that notebook is running inside the virtual enivronment we created.

In [47]:
import sys
print("Python executable:", sys.executable)

Python executable: c:\Users\julyz\Downloads\NLP Project\.venv\Scripts\python.exe


### Step 1.2 - Force PyTorch Only
Tells transformers to use PyTorch and never try to import TensorFlow because earlier it caused DDL/package errors.

In [2]:
import os
os.environ["USE_TF"]             = "0"
os.environ["USE_TORCH"]          = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"

print("Environment set - using PyTorch only.")

Environment set - using PyTorch only.


### Step 1.3 - Load API Keys from .env

In [3]:
from dotenv import load_dotenv

load_dotenv()

QDRANT_URL     = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
GROQ_API_KEY   = os.getenv("GROQ_API_KEY")

print("QDRANT_URL loaded:     ", QDRANT_URL is not None)
print("QDRANT_API_KEY loaded: ", QDRANT_API_KEY is not None)
print("GROQ_API_KEY loaded:   ", GROQ_API_KEY is not None)

QDRANT_URL loaded:      True
QDRANT_API_KEY loaded:  True
GROQ_API_KEY loaded:    True


### Step 1.4 - Import All Dependencies

In [4]:
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from groq import Groq

print("All libraries imported successfully.")

c:\Users\julyz\Downloads\NLP Project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully.


---
## 2 - Load and Explore the Dataset

### Step 2.1 - Load the Dataset

We load the Mental Health Counseling Conversations dataset from HuggingFace.

Two columns:
- Context: the user question or concern
- Response: the counselor answer

We combine both into combined_text for embedding.
This allows the retriever to match on both the problem and the answer.

In [48]:
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")
df = dataset.to_pandas()

print(f"Total rows: {len(df)}")

Total rows: 3512


### Step 2.2 - Explore the Structure

In [6]:
print("Shape:  ", df.shape)
print("Columns:", df.columns.tolist())

Shape:   (3512, 2)
Columns: ['Context', 'Response']


### Step 2.3 - Preview Sample Rows

In [7]:
df.head(3)

,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...


In [49]:
print("Context (User Question):")
print(df['Context'][0])
print()
print("Response (Counselor Answer):")
print(df['Response'][0])

Context (User Question):
I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.
   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.
   How can I change my feeling of being worthless to everyone?

Response (Counselor Answer):
If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media.  Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselv

### Step 2.4 - Check for Missing Values
I checked missing values because empty text should not be embedded or stored in the vector database.

In [9]:
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
Context     0
Response    0
dtype: int64


### Step 2.5 - Check Length Distribution of Both Columns

Findings:
- Context min = 25 chars - acceptable
- Response min = 0 chars - some empty responses exist, dropped in next steps.
- Long responses are fine - sentence transformer handles truncation automatically

In [10]:
df['context_length']  = df['Context'].astype(str).str.len()
df['response_length'] = df['Response'].astype(str).str.len()

print("Context length statistics:")
print(df['context_length'].describe().round(0))

print("\nResponse length statistics:")
print(df['response_length'].describe().round(0))

Context length statistics:
count    3512.0
mean      283.0
std       246.0
min        25.0
25%       147.0
50%       232.0
75%       348.0
max      2703.0
Name: context_length, dtype: float64

Response length statistics:
count     3512.0
mean      1026.0
std       1011.0
min          0.0
25%        532.0
50%        836.0
75%       1272.0
max      32739.0
Name: response_length, dtype: float64


---
## 3 - Data Preprocessing

We preformed these preprocessing steps:
1. Drop rows where Response is empty
2. Strip whitespace from both columns
3. Combine Context + Response into combined_text
4. Reset index and confirm final shape

### Step 3.1 - Drop Empty Responses

Response min = 0 means some rows have no content.

In [11]:
rows_before = len(df)

df = df[df['Response'].astype(str).str.strip() != '']
df = df[df['Response'].astype(str).str.strip() != 'nan']

rows_after = len(df)

print(f"Rows before:  {rows_before}")
print(f"Rows after:   {rows_after}")
print(f"Rows dropped: {rows_before - rows_after}")

Rows before:  3512
Rows after:   3508
Rows dropped: 4


### Step 3.2 - Clean Text
Strip whitespace

In [50]:
df['Context']  = df['Context'].astype(str).str.strip()
df['Response'] = df['Response'].astype(str).str.strip()

### Step 3.3 - Combine Context and Response into One Field

I combined Context and Response so each vector represents both the user concern and the counseling guidance.

In [13]:
df['combined_text'] = df['Context'] + " " + df['Response']

print("Combined text field created.")
print("\nExample (first row, first 300 chars):")
print(df['combined_text'].iloc[0][:300], "...")

Combined text field created.

Example (first row, first 300 chars):
I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.
   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.
   How can I change my feeling of  ...


### Step 3.4 - Reset Index and Confirm Final Dataset
I reset the index so each cleaned row maps clearly to its embedding and Qdrant point ID.

In [14]:
df = df.reset_index(drop=True)

print(f"Final dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nReady to embed {len(df)} chunks.")

Final dataset shape: (3508, 5)
Columns: ['Context', 'Response', 'context_length', 'response_length', 'combined_text']

Ready to embed 3508 chunks.


---
## 4 - Embedding
We encode every combined_text chunk into a dense vector using all-MiniLM-L6-v2.

Why this model:
- Pretrained - no training needed from us
- Produces 384-dimensional vectors
- Fast on CPU
- Standard model for RAG semantic search


### Step 4.1 - Load the Embedding Model

In [15]:
# First run downloads the model (~80MB). After that it loads from cache.
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Model loaded.")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

c:\Users\julyz\Downloads\NLP Project\.venv\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model loaded.
Embedding dimension: 384


### Step 4.2 - Embed All Chunks

batch_size=64 processes 64 texts at a time to manage memory efficiently.
This may take a few minutes on CPU.

In [16]:
chunks = df['combined_text'].tolist()

embeddings = model.encode(
    chunks,
    batch_size=64,
    show_progress_bar=True
)

print(f"\nEmbedding complete.")
print(f"Total vectors: {len(embeddings)}")
print(f"Vector shape:  {embeddings.shape}")

Batches: 100%|██████████| 55/55 [01:27<00:00,  1.58s/it]


Embedding complete.
Total vectors: 3508
Vector shape:  (3508, 384)


### Step 4.3 - Verify a Single Vector

In [17]:
print(f"First vector - dimension: {len(embeddings[0])}")
print(f"First 5 values: {embeddings[0][:5]}")

First vector - dimension: 384
First 5 values: [ 0.04152797  0.00503538 -0.0185178   0.04446397 -0.00131953]


---
## 5 - Qdrant Setup and Upload

We connect to Qdrant Cloud, create a collection, and upload all 3508 vectors.
After this phase the knowledge base lives in the cloud and is ready to search.

### Step 5.1 - Connect to Qdrant Cloud

In [18]:
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)

print("Connected to Qdrant Cloud.")

Connected to Qdrant Cloud.


### Step 5.2 - Create a Collection

A collection in Qdrant is like a table in a database - it stores all our vectors.

- size=384 matches our embedding dimension from all-MiniLM-L6-v2
- Distance.COSINE measures similarity between vectors using cosine similarity

In [ ]:
COLLECTION_NAME = "mental_health_chunks"

existing = [c.name for c in client.get_collections().collections]

if COLLECTION_NAME in existing:
    print(f"Collection '{COLLECTION_NAME}' already exists. Using existing collection.")
else:
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=384,
            distance=Distance.COSINE
        )
    )
    print(f"Collection '{COLLECTION_NAME}' created.")

Collection 'mental_health_chunks' created.


### Step 5.3 - Upload Vectors to Qdrant

We upload each vector together with its original text as payload.
The payload lets us retrieve the actual text after a search - not just the vector ID.

In [21]:
import time

batch_size = 50
total_rows = len(df)

uploaded = 0

for start_idx in range(0, total_rows, batch_size):
    end_idx = min(start_idx + batch_size, total_rows)

    batch_points = []

    for i in range(start_idx, end_idx):
        batch_points.append(
            PointStruct(
                id=int(i),
                vector=embeddings[i].tolist(),
                payload={
                    "context": str(df["Context"].iloc[i]),
                    "response": str(df["Response"].iloc[i])
                }
            )
        )

    client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch_points,
        wait=True
    )

    uploaded += len(batch_points)
    print(f"Uploaded {uploaded}/{total_rows} vectors")

    time.sleep(0.2)

print("All vectors uploaded to Qdrant.")

Uploaded 50/3508 vectors
Uploaded 100/3508 vectors
Uploaded 150/3508 vectors
Uploaded 200/3508 vectors
Uploaded 250/3508 vectors
Uploaded 300/3508 vectors
Uploaded 350/3508 vectors
Uploaded 400/3508 vectors
Uploaded 450/3508 vectors
Uploaded 500/3508 vectors
Uploaded 550/3508 vectors
Uploaded 600/3508 vectors
Uploaded 650/3508 vectors
Uploaded 700/3508 vectors
Uploaded 750/3508 vectors
Uploaded 800/3508 vectors
Uploaded 850/3508 vectors
Uploaded 900/3508 vectors
Uploaded 950/3508 vectors
Uploaded 1000/3508 vectors
Uploaded 1050/3508 vectors
Uploaded 1100/3508 vectors
Uploaded 1150/3508 vectors
Uploaded 1200/3508 vectors
Uploaded 1250/3508 vectors
Uploaded 1300/3508 vectors
Uploaded 1350/3508 vectors
Uploaded 1400/3508 vectors
Uploaded 1450/3508 vectors
Uploaded 1500/3508 vectors
Uploaded 1550/3508 vectors
Uploaded 1600/3508 vectors
Uploaded 1650/3508 vectors
Uploaded 1700/3508 vectors
Uploaded 1750/3508 vectors
Uploaded 1800/3508 vectors
Uploaded 1850/3508 vectors
Uploaded 1900/3508 ve

### Step 5.4 - Verify Upload

In [22]:
count = client.get_collection(COLLECTION_NAME).points_count
print(f"Vectors in collection: {count}")
assert count == len(df), "Upload count mismatch - something went wrong."
print("Upload verified successfully.")

Vectors in collection: 3508
Upload verified successfully.


---
## 6 - Query Pipeline

This is the core RAG function that runs every time a user asks a question.

Flow:
1. Embed the user question using the same sentence transformer
2. Search Qdrant for the top-k most similar chunks
3. Build a prompt with the retrieved context and the question
4. Call Groq API and get a grounded empathetic response
5. Return the answer

### Step 6.1 - Initialize Groq Client

In [34]:
groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq client initialized.")

Groq client initialized.


### Step 6.2 - Define the RAG Query Function

This function takes a user question and an optional emotion label from Module 2,
retrieves relevant chunks from Qdrant, and generates a response using Groq.

In [35]:
def rag_query(question, emotion=None, top_k=3):
    """
    RAG pipeline:
    1. Embed the user question
    2. Search Qdrant for top-k similar chunks
    3. Build a prompt using retrieved Context + Response
    4. Call Groq and return the generated answer
    """

    # Step 1 - Embed the question
    query_vector = model.encode(question).tolist()

    # Step 2 - Search Qdrant using query_points
    search_result = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True
    )

    results = search_result.points

    # Step 3 - Build retrieved context using both original user concern and counselor response
    retrieved_chunks = []

    for i, hit in enumerate(results):
        retrieved_chunks.append(
            f"[Chunk {i+1}]\n"
            f"Similarity score: {hit.score}\n\n"
            f"Original user concern:\n{hit.payload['context']}\n\n"
            f"Counselor response:\n{hit.payload['response']}"
        )

    retrieved_context = "\n\n---\n\n".join(retrieved_chunks)

    # Step 4 - Add emotion note if available
    emotion_note = f"The detected emotion is: {emotion}.\n" if emotion else ""

    # Step 5 - Build prompt
    system_prompt = (
        "You are a supportive mental health assistant. "
        "Use only the retrieved context to answer the user. "
        "Be empathetic, calm, and clear. "
        "Do not diagnose the user. "
        "Do not claim to be a therapist or doctor. "
        "If the context is not enough, say that clearly and give general supportive guidance."
    )

    user_prompt = (
        f"{emotion_note}"
        f"Retrieved context from the counseling knowledge base:\n\n"
        f"{retrieved_context}\n\n"
        f"User question:\n{question}\n\n"
        f"Answer:"
    )

    # Step 6 - Call Groq
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.3,
        max_tokens=800
    )

    return response.choices[0].message.content


print("RAG query function defined.")

RAG query function defined.


---
## 7 - Testing

We run sample mental health questions through the full pipeline
to confirm everything works end to end.

### Step 7.1 - Test Question 1 (no emotion)

In [36]:
question_1 = "I feel worthless and I don't know how to stop these thoughts."

answer_1 = rag_query(question=question_1)

print("Question:", question_1)
print()
print("Answer:")
print(answer_1)

Question: I feel worthless and I don't know how to stop these thoughts.

Answer:
I’m really sorry you’re feeling this way. It can feel overwhelming when thoughts of worthlessness keep coming up, but you’re not alone, and there are small steps you can try to start easing those thoughts.

---

### 1. **Notice the pattern**

- **When do the thoughts arise?**  
  Pay attention to the time of day, the situation, or the people around you that seem to bring them up.  
- **What’s the exact wording?**  
  Write down the phrase that pops into your head (“I’m worthless,” “I shouldn’t be here,” etc.). Seeing it on paper can make it feel a bit more distant.

---

### 2. **Explore the source (in a gentle way)**

- **When did it start?**  
  Think about a time—maybe a childhood event or a recent experience—when you first noticed this feeling.  
- **Who’s voice is it?**  
  Often the negative voice is someone else’s expectation or a harsh inner critic. Identifying it can help you see it as a separate 

### Step 7.2 - Test Question 2 (with emotion label from Module 2)

In [37]:
question_2 = "I have been having panic attacks and I don't know what to do."
emotion_2  = "anxiety"

answer_2 = rag_query(question=question_2, emotion=emotion_2)

print("Question:", question_2)
print("Emotion: ", emotion_2)
print()
print("Answer:")
print(answer_2)

Question: I have been having panic attacks and I don't know what to do.
Emotion:  anxiety

Answer:
I’m really sorry you’re going through this. Panic attacks can feel overwhelming, but there are steps you can take right now to help ease the intensity and start feeling a bit more in control.

**1. Ground yourself in the present moment**  
- Try a simple breathing exercise: inhale slowly for 4 counts, hold for 4, exhale for 6. Repeat a few times.  
- Notice something around you—touch a chair, feel the texture of a blanket, or listen to a nearby sound. Shifting focus can help reduce the racing thoughts that often accompany a panic attack.

**2. Use calming tools you can do on your own**  
- Guided meditation or a short yoga routine can lower heart rate and calm the nervous system. There are many free apps or YouTube videos that walk you through a few minutes of mindfulness.  
- Keep a small “calm‑kit” handy: a stress ball, a favorite song, or a comforting scent. When you feel a panic attac

### Step 7.3 - Test Question 3 (with emotion label)

In [38]:
question_3 = "I feel very depressed and I have no motivation to do anything."
emotion_3  = "sadness"

answer_3 = rag_query(question=question_3, emotion=emotion_3)

print("Question:", question_3)
print("Emotion: ", emotion_3)
print()
print("Answer:")
print(answer_3)

Question: I feel very depressed and I have no motivation to do anything.
Emotion:  sadness

Answer:
I’m really sorry you’re feeling so overwhelmed right now. It can be hard to see a way forward when everything feels heavy, but you don’t have to face this alone. Below are a few gentle steps that might help you start to lift some of that weight, one small piece at a time.

---

### 1. **Start with a tiny, doable action**

Pick one thing that feels almost effortless—maybe:

- **Take a short walk** (even a few minutes around the block).  
- **Drink a glass of water** or have a light snack.  
- **Set a timer for 5 minutes** and do a breathing exercise or stretch.

When you accomplish that tiny task, give yourself a brief moment of acknowledgment. It’s a small win that can build momentum.

---

### 2. **Ground yourself in the present**

- **Mindful breathing**: Inhale for 4 counts, hold for 4, exhale for 4. Repeat a few times.  
- **Grounding technique**: Notice five things you can see, four

---
## 8 - Crisis Safety Check and Final Chatbot Function

Here we add:
1. A simple keyword-based crisis detection rule
2. A final chatbot function that wraps crisis check + RAG

**Important:** This is NOT a medical classifier.
It only catches obvious crisis phrases so the chatbot
returns a direct safety message instead of a RAG response.

### Step 8.1 - Crisis Detection Function

A simple keyword check. If the user message contains any obvious crisis phrase,
we skip RAG entirely and return a direct safety response.

This is intentionally simple. It covers common explicit phrases only.
It is not a substitute for real clinical risk assessment.

In [39]:
CRISIS_KEYWORDS = [
    "suicide",
    "kill myself",
    "end my life",
    "self harm",
    "self-harm",
    "hurt myself",
    "don't want to live",
    "do not want to live",
    "want to die",
    "take my own life",
    "no reason to live",
]

CRISIS_RESPONSE = (
    "I hear you, and I am very concerned about your safety right now. "
    "Please reach out to a crisis support line immediately:\n\n"
    "- **International Association for Suicide Prevention:** https://www.iasp.info/resources/Crisis_Centres/\n"
    "- **Crisis Text Line (US):** Text HOME to 741741\n"
    "- **Befrienders Worldwide:** https://www.befrienders.org\n\n"
    "You are not alone. A trained counselor is available to talk with you right now. "
    "If you are in immediate danger, please call your local emergency services."
)

def is_crisis(text: str) -> bool:
    """Return True if the text contains any obvious crisis phrase."""
    text_lower = text.lower()
    return any(kw in text_lower for kw in CRISIS_KEYWORDS)

print("Crisis detection function defined.")
print(f"Monitoring {len(CRISIS_KEYWORDS)} crisis keywords.")


Crisis detection function defined.
Monitoring 11 crisis keywords.


### Step 8.2 - Final Chatbot Function

This is the single entry point for Module 4 at runtime.

Input:
- question (str): the user's message
- emotion (str, optional): emotion label from Module 2

Flow:
1. Check for crisis language → return safety message
2. Otherwise → call rag_query and return the answer

This function is designed to be easy to copy into the future app/script.

In [40]:
def mental_health_chatbot(question: str, emotion: str = None) -> str:
    """
    Final Module 4 chatbot function.

    Args:
        question: user message
        emotion:  optional emotion label from Module 2 (e.g. 'anxiety', 'sadness')

    Returns:
        Safety message if crisis detected, otherwise RAG-generated answer.
    """
    if is_crisis(question):
        return CRISIS_RESPONSE

    return rag_query(question=question, emotion=emotion)

print("Final chatbot function defined.")


Final chatbot function defined.


---
## 9 - Final Tests

Four tests that cover the complete Module 4 behavior:
1. Anxiety question (RAG)
2. Depression question (RAG)
3. Stress / overthinking question (RAG)
4. Crisis question (safety message - no RAG call)

### Test 1 - Anxiety Question

In [41]:
q1 = "I keep having panic attacks and I cannot calm down. What should I do?"
e1 = "anxiety"

print("=" * 60)
print("TEST 1 - Anxiety")
print("Question:", q1)
print("Emotion: ", e1)
print("-" * 60)
a1 = mental_health_chatbot(question=q1, emotion=e1)
print(a1)
print("=" * 60)


TEST 1 - Anxiety
Question: I keep having panic attacks and I cannot calm down. What should I do?
Emotion:  anxiety
------------------------------------------------------------
I’m really sorry you’re feeling so overwhelmed right now. Panic attacks can feel like a sudden, intense wave that’s hard to stop, but there are a few things you can try to help bring that wave down a bit.

**1. Ground yourself with breathing.**  
A simple, quick technique is to breathe in for a count of five, hold for five, and breathe out for five. Repeat this a few times. The goal is to slow your breathing and give your body a chance to reset. If you can, do this before the panic starts or as soon as you notice the first signs.

**2. Use a short mindfulness cue.**  
Pick one sensory focus—like the feel of your feet on the floor, the sound of a nearby clock, or the texture of a piece of fabric. When you feel the panic rising, gently bring your attention to that sensation. It can help pull you back from the racin

### Test 2 - Depression Question

In [43]:
q2 = "I feel completely hopeless and I have no energy to do anything."
e2 = "sadness"

print("=" * 60)
print("TEST 2 - Depression")
print("Question:", q2)
print("Emotion: ", e2)
print("-" * 60)
a2 = mental_health_chatbot(question=q2, emotion=e2)
print(a2)
print("=" * 60)


TEST 2 - Depression
Question: I feel completely hopeless and I have no energy to do anything.
Emotion:  sadness
------------------------------------------------------------
I’m really sorry you’re feeling so drained and hopeless right now. It can feel overwhelming when everything seems to weigh on you, but you’re not alone in this, and there are small, gentle steps you can try that might help lift that heaviness a bit.

**1. Start with the basics of self‑care.**  
- **Sleep:** Even a short, consistent bedtime routine can make a difference. Try setting a regular wake‑up time, dimming lights an hour before bed, and limiting screens. If you’re having trouble falling asleep, a brief 5‑minute breathing exercise or a short stretch can help calm the mind.  
- **Nutrition & hydration:** Eating regular, balanced meals and drinking water can support your energy levels, even if you don’t feel like cooking. A simple snack—like a piece of fruit or a handful of nuts—can sometimes give a quick boost.

### Test 3 - Stress / Overthinking Question

In [44]:
q3 = "My mind never stops. I overthink everything and it is exhausting."
e3 = "stress"

print("=" * 60)
print("TEST 3 - Stress / Overthinking")
print("Question:", q3)
print("Emotion: ", e3)
print("-" * 60)
a3 = mental_health_chatbot(question=q3, emotion=e3)
print(a3)
print("=" * 60)


TEST 3 - Stress / Overthinking
Question: My mind never stops. I overthink everything and it is exhausting.
Emotion:  stress
------------------------------------------------------------
I hear how exhausting it feels when your thoughts keep racing—like a loop that never stops. It’s completely understandable that this would leave you feeling drained and anxious. You’re not alone; many people experience the same kind of over‑thinking, especially when the mind keeps circling around big questions or worries.

Here are a few gentle strategies that might help you pause the cycle and bring a bit of calm into the present moment:

| Strategy | What it does | How to try it |
|----------|--------------|---------------|
| **Grounding / “5‑4‑3‑2‑1”** | Ties your awareness to the here and now. | Notice 5 things you can see, 4 you can touch, 3 you can hear, 2 you can smell, 1 you can taste. |
| **Mindful breathing** | Gives the brain a simple, rhythmic focus. | Breathe in for 4 counts, hold 4, exhale 

### Test 4 - Crisis Question

This question contains a crisis phrase.
The chatbot must return the safety message directly.
The RAG pipeline must NOT be called.

In [45]:
q4 = "I want to kill myself. I cannot take it anymore."

print("=" * 60)
print("TEST 4 - Crisis Detection")
print("Question:", q4)
print("-" * 60)
a4 = mental_health_chatbot(question=q4)
print(a4)
print()
print("Crisis detected correctly:", is_crisis(q4))
print("=" * 60)


TEST 4 - Crisis Detection
Question: I want to kill myself. I cannot take it anymore.
------------------------------------------------------------
I hear you, and I am very concerned about your safety right now. Please reach out to a crisis support line immediately:

- **International Association for Suicide Prevention:** https://www.iasp.info/resources/Crisis_Centres/
- **Crisis Text Line (US):** Text HOME to 741741
- **Befrienders Worldwide:** https://www.befrienders.org

You are not alone. A trained counselor is available to talk with you right now. If you are in immediate danger, please call your local emergency services.

Crisis detected correctly: True
